# Downloading/processing OC-CCI and ERA5 data for Antarctic coastal blooms project

In [1]:
from numpy import *
import xarray as xr
import pandas as pd
import dask
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import os
import shutil
import platform
import warnings
import requests
import urllib3
from ftplib import FTP
import getpass
import subprocess
import time
import cdsapi

In [2]:
# directories
data_dir = '/dat1/ethancc/Data/Coastal blooms project/'
chl_dir = data_dir + 'OC_CCI/'
chl_orig_dir = chl_dir + 'Original_files/'
era5_dir = data_dir + 'ERA5/'
era5_mo_means_dir = data_dir + 'ERA5_monthly_means/'

## Functions for downloading/loading data

In [ ]:
def single_file(url_dir,filename,save_to,ftp_root=False,overwrite=False,verbose=True,auth=None,cert=True,
                nasa_auth_session=None):
    """ Downloads and saves a file from a given URL.

    Notes:
        - For HTTP downloads, if '404 file not found' error returned, function will return without
          downloading anything.
        - For FTP downloads, if given filename doesn't exist in directory, function will return without
          downloading anything.
    
    Args:
        url_dir: URL up to the filename, including ending slash
            NOTE: for ftp servers, include URL after the root, without starting slash
        filename: filename, including suffix
        save_to: directory path, including trailing slash but not including the filename
        ftp_root: root URL of ftp server without preamble (ftp://) or ending slash, or 'False' if using HTTP
        overwrite: False (default) to leave existing files in place; True to overwrite
            (note: doesn't explicitly/separately delete existing file before downloading new file)
        cert: True (default) to validate SSL certificate; False to ignore validity of SSL certificate
        nasa_auth_session: if NASA Earthdata authentication required, pass 'sessions' instance from df.nasa_auth()
            (note: not relevant in this notebook)
    
    """
    starting_dir = os.getcwd()

    if cert is False:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    try:
        if starting_dir is not save_to:
            os.chdir(save_to)
        if filename in os.listdir():
            if not overwrite:
                if verbose: print('>>> File ' + filename + ' already exists. Leaving current version.')
                return
            else:
                if verbose: print('>>> File ' + filename + ' already exists. Overwriting with new version.')

        if not ftp_root:
            full_url = url_dir + filename

            def get_func(url, stream=True, auth_key=None, verify=cert):
                try:
                    if nasa_auth_session is None:
                        return requests.get(url, stream=stream, auth=auth_key, verify=cert)
                    else:
                        return nasa_auth_session.get(url,stream=stream,auth=auth_key,verify=cert)
                except requests.exceptions.ConnectionError as error_tag:
                    print('Error connecting:', error_tag)
                    # note: this solution is super hacky and bad practice,
                    #       see https://stackoverflow.com/questions/16511337/correct-way-to-try-except-using-python-requests-module
                    time.sleep(1)
                    return get_func(url, stream=stream, auth_key=auth_key, verify=cert)

            response = get_func(full_url, stream=True, auth_key=auth, verify=cert)

            if response.status_code == 404:
                if verbose: print('>>> File ' + filename + ' returned 404 error during download.')
                return
            with open(filename,'wb') as out_file:
                shutil.copyfileobj(response.raw, out_file)
            del response
        else:
            ftp = FTP(ftp_root)
            ftp.login();
            ftp.cwd(url_dir);
            contents = ftp.nlst();
            if filename in contents:
                local_file = open(filename,'wb')
                ftp.retrbinary('RETR ' + filename, local_file.write);
                local_file.close()
            ftp.quit();
    finally:
        os.chdir(starting_dir)

In [3]:
def era5(area=[-50,-180,-80,180],years=[str(yr) for yr in range(2003,2024+1)],variables=[None],download_dir=None,
         batch=False,legacy_netcdf=False):
    """ Submits CDS API request to retrieve ERA5 reanalysis fields (0.25° x 0.25° grid) as netCDF file.
    
    NOTE: if downloading variables not on the example list below, then make sure to update their GRIB1/GRIB2 variable name
          and 'long_name' attribute correspondence in ldp.load_era5(). To do this, find their GRIB1 names on the ERA5 data
          documentation (https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation) and click the link to find
          the GRIB2 names in the ECMWF parameter database.

    Arguments:
        'area': [North, West, South, East] in °N or °E (-180 to 180)
        'years': list of years given as strings
        'variables': list of variables given as strings; the following are examples:
            '10m_u_component_of_wind'
            '10m_v_component_of_wind'
            '2m_dewpoint_temperature'
            '2m_temperature'
            'sea_surface_temperature' [*]
            'skin_temperature' [*]
            'mean_eastward_turbulent_surface_stress' [*]
            'mean_northward_turbulent_surface_stress' [*]
            'mean_evaporation_rate'
            'mean_total_precipitation_rate'
            'mean_snowfall_rate'
            'surface_pressure'
            'mean_sea_level_pressure' [*]
            'sea_ice_cover' [NOTE: now called 'sea_ice_area_fraction' with short name 'ci', not 'siconc', but old names may still work]
            'surface_latent_heat_flux' [*]
            'surface_sensible_heat_flux' [*]
            'surface_net_solar_radiation' [*]
            'surface_net_thermal_radiation' [*]
        'download_dir': None, to queue request and download from web browser
                        or directory path (including trailing backslash) to download directly into (for small requests)
        'batch': False (default) to display status of request sending / queued / running sequence
                 True to submit request, then disconnect from ECMWF to allow additional submissions (e.g., in a loop)
                 (note: if True, <download_dir> will default to behavior of None)
        'legacy_netcdf': False (default) to ask ECMWF to process files using current GRIB-to-netCDF4 converter
                         True to ask ECMWF to process files using legacy GRIB-to-netCDF3 converter; see ldp.load_era5() for more details

    Documentation:
        - general ERA5 documentation: https://confluence.ecmwf.int/display/CKB/ERA5+data+documentation
        - how to download: https://confluence.ecmwf.int/display/CKB/How+to+download+ERA5
        - parameter database: https://apps.ecmwf.int/codes/grib/param-db

    Cite using:
        Hersbach et al. (2020). The ERA5 global reanalysis, Quarterly Journal of the Royal Meteorological Society, TBD.
            doi:10.1002/qj.3803. https://onlinelibrary.wiley.com/doi/abs/10.1002/qj.3803
        Copernicus Climate Change Service (C3S) (2017). ERA5: Fifth generation of ECMWF atmospheric reanalyses of the
            global climate. Copernicus Climate Change Service Climate Data Store (CDS), date of access.
            https://cds.climate.copernicus.eu/cdsapp#!/home

    Important notes:
        - local installation of CDS API key is required! see details: https://cds.climate.copernicus.eu/api-how-to
        - for most (large) requests, after "Request is queued" appears, cancel using Ctrl-C and download from here:
          https://cds.climate.copernicus.eu/requests?tab=all
        - note that it seems that including years containing both validated ERA5 and preliminary ERA5T data within
          multiyear downloads is problematic, causing processing failures, so split up downloads, e.g. 2012-2018 then 2019
        - on MacOS, consider using a stand-alone download manager app, such as Download Shuttle, instead of a web
          browser download interface
        - on Linux terminal, download files from ECMWF CDS in background using wget -bqc <URL from ECMWF portal>
        - limit for requests used to be 120,000 "items" (where 1 hourly spatial field of 1 variable = 1 item), but may have decreased recently
        - known biases include high instantaneous surface stress; workaround is to use accumulated surface stress
          fields, as described here: https://confluence.ecmwf.int/display/CKB/ERA5+instantaneous+surface+stress+and+friction+velocity+over+the+oceans

    """
    time_str = str(datetime.now().date()) + '-' + str(datetime.now().time()).replace(':','-').replace('.','-')
    if download_dir is None: filepath = os.getcwd() + '/era5_download_{}.nc'.format(time_str)
    else:                    filepath = download_dir + 'era5_download_{}.nc'.format(time_str)

    if batch: filepath = None

    if not batch:
        c = cdsapi.Client()
    if batch:
        c = cdsapi.Client(wait_until_complete=False,delete=False,forget=True)
        
    if legacy_netcdf:
        netcdf_format = 'netcdf_legacy'
    else:
        netcdf_format = 'netcdf'
        
    c.retrieve('reanalysis-era5-single-levels', {
            'product_type':['reanalysis'],
            'format':netcdf_format,
            'download_format':'unarchived',
            'variable': variables,
            'year': years,
            'month': [
                '01','02','03',
                '04','05','06',
                '07','08','09',
                '10','11','12'],
            'day': [
                '01','02','03',
                '04','05','06',
                '07','08','09',
                '10','11','12',
                '13','14','15',
                '16','17','18',
                '19','20','21',
                '22','23','24',
                '25','26','27',
                '28','29','30',
                '31'],
            'time': [
                '00:00','01:00','02:00',
                '03:00','04:00','05:00',
                '06:00','07:00','08:00',
                '09:00','10:00','11:00',
                '12:00','13:00','14:00',
                '15:00','16:00','17:00',
                '18:00','19:00','20:00',
                '21:00','22:00','23:00'],
            'area': area,
    }, filepath)
    
    
def load_era5(data_dir,process_and_export=False,datetime_range=None,lat_range=None,lon_range=None,
              time_chunk=None,lat_chunk=None,lon_chunk=None,rechunk=False,use_grib2_names=False):
    """ Opens ERA5 reanalysis data files downloaded in netCDF format.

    Args:
        data_dir: directory of all data files (all files ending in '.nc' will be loaded)
        process_and_export: False (default) to simply load all files, assuming they've been processed already
                            True to do two processing tasks:
                                (1) assess all available files and merge validated ERA5 data with preliminary
                                    ERA5T data when necessary, then export the resulting file and move the original
                                (2) assess all available files and split files with more than one variable into
                                    separate exported files, then move the original to the "To delete" directory
            note: no Dataset will be returned if True
        datetime_range: None or [Datetime0,Datetime1] or [Datestring0,Datestring1] to subset fields
            note: to slice with open right end, e.g., use [Datetime0,None]
            note: selection is generous, so ['2016-1-1','2016-1-1'] will include all hours on January 1, 2016
            note: example of Datestring: '2016-1-1-h12' or '2016-01-01 12:00'
        lat_range: None or [lat_N,lat_S] to subset fields (!!! - descending order - !!!)
        lon_range: None or [lon_W,lon_E] to subset fields
        time_chunk, lat_chunk, lon_chunk: specify chunk sizes (otherwise use lat_chunk=20, lon_chunk=100, and
                                          compute time_chunk such that chunks are 1-5 MB each)
        rechunk: re-chunk with specified or computed chunk sizes (useful if time indices vary across files)
        use_grib2_names: False to use GRIB1 naming conventions, renaming all variables (and their 'long_name' attribute)
                             whose names were changed in ECMWF's migration from GRIB1 to GRIB2
                         True to use GRIB2 naming conventions, similarly renaming those changed from GRIB1

    Returns:
        all_data: xarray Dataset with coordinates (time,lats,lons); examples of accessing/slicing follow:
            all_data.loc[dict(time='2016-1-1')]                            to extract without slicing
            all_data.sel(lats=slice(-60,-70))                              to slice all variables
            all_data['skt'].values                                         to convert to NumPy array in memory
            all_data['skt'][0,:,:]  or  all_data.isel(time=0)              to slice data using indices (t=0)
            all_data['skt'].loc['2016-1-1':'2016-2-1',-60:-70,0:10]        to slice data using values (not recommended)
            all_data['skt']['latitude']                                    to get view of 1-D coordinate
            all_data['skt']['time']                                        NumPy Datetime coordinate
            all_data['skt']['doy']                                         fractional day-of-year coordinate
            pd.to_datetime(all_data['skt']['time'].values)                 useable Datetime version of the above
            all_data['skt'].attrs['units']
            all_data['skt'].attrs['long_name']

    Note: as shown above, 'doy' (fractional day-of-year) is included as a secondary coordinate with dimension 'time'.
    
    Note: after ECMWF's migration to a new GRIB to netCDF4 converter in late 2024, xarray I/O with the resulting netCDF4 files is extremely slow
          compared to the previous netCDF3 files. This seems to be related to poor chunking decisions in their converter settings (see threads
          linked below). This slow I/O be confirmed by timing simple .sel().values operations for a subset of data (see below). The only solution if
          staying with the new netCDF4 files would likely be using ncks or a similar utility to pre-process all ERA5 files with a more optimized
          chunking scheme before loading in xarray. An alternative could be requesting the original GRIB files from ECMWF, which xarray can load.
          In the interim, ECMWF has provided an option to request the legacy netCDF3 files, which is an option in dlp.era5(). This option may
          eventually be removed by ECMWF (and the netCDF3 files are "not supported" in any case) so this issue should be revisited at a later date.
          Likely related to this, time_chunk values computed in this routine for the new netCDF4 files were approx. 800, while they should be ~300
          for optimal performance.
          
          TESTING ROUTINE: the following timing test for any date of data should give an elapsed time of 0.2-0.4 seconds:
          >>> timer_start = time.time()
          >>> era5['mtpr'].sel(time='2025-03-28T00').sel(lons=(-50,50),lats=(-70,-65)).mean().values
          >>> timer_end = time.time()
          >>> print('elapsed time: {0:.1f} s'.format(timer_end - timer_start))
          
          - ECMWF info page: https://confluence.ecmwf.int/display/CKB/GRIB+to+netCDF+conversion+on+new+CDS+and+ADS+systems#GRIBtonetCDFconversiononnewCDSandADSsystems-ERA5
          - See comment by Tristan Schuler on this thread: https://forum.ecmwf.int/t/changes-to-grib-to-netcdf-converter-on-cds-beta-ads-beta/4322/35?page=2
          - See comment by Kenneth Bowman on this thread: https://forum.ecmwf.int/t/forthcoming-update-to-the-format-of-netcdf-files-produced-by-the-conversion-of-grib-data-on-the-cds/7772/4
          - And see this thread: https://code.mpimet.mpg.de/boards/2/topics/16082

    The following derived quantities are calculated here, to be evaluated lazily using Dask:
        'q2m': 2-m specific humidity from 'msl' and 'd2m'
        'si10': 10-m wind speed from 'u10' and 'v10'

    """
    name_list = array([('mer','avg_ie','Mean evaporation rate','Time-mean moisture flux'),
                       ('mtpr','avg_tprate','Mean total precipitation rate','Time-mean total precipitation rate'),
                       ('msr','avg_tsrwe','Mean snowfall rate','Time-mean total snowfall rate water equivalent'),
                       ('metss','avg_iews','Mean eastward turbulent surface stress','Time-mean eastward turbulent surface stress'),
                       ('mntss','avg_inss','Mean northward turbulent surface stress','Time-mean northward turbulent surface stress'),
                       ('slhf','slhf','Surface latent heat flux','Time-integrated surface latent heat net flux'),
                       ('sshf','sshf','Surface sensible heat flux','Time-integrated surface sensible heat net flux'),
                       ('ssr','ssr','Surface net solar radiation','Surface net short-wave (solar) radiation'),
                       ('str','str','Surface net thermal radiation','Surface net long-wave (thermal) radiation')])
    grib_rename = pd.DataFrame(data=name_list,columns=['GRIB1_var','GRIB2_var','GRIB1_long','GRIB2_long'])
    if use_grib2_names: grib_rename = grib_rename.set_index('GRIB1_var'); grib_str = 'GRIB2'  # prepare to change any GRIB1 names
    else:               grib_rename = grib_rename.set_index('GRIB2_var'); grib_str = 'GRIB1'  # prepare to change any GRIB2 names

    # list all files in directory
    all_filenames = os.listdir(data_dir)
    netcdf_filenames = []
    for filename in all_filenames:
        if '.nc' in filename: netcdf_filenames.append(filename)    # ignore subdirectories and '.DS_Store' file
    all_filenames = netcdf_filenames

    if time_chunk is None and lat_chunk is None and lon_chunk is None:
        # compute chunk size for Dask; aiming for each chunk to be between 1-5 MB
        # total chunk size computed here as (0.0025 GB) * len(data.time) / filesize, using first file in directory
        first_filename = all_filenames[0]
        first_file_size = os.path.getsize(data_dir + first_filename) / 1E9     # in GB
        first_file_data = xr.open_dataset(data_dir + first_filename,chunks={})
        if 'valid_time' in first_file_data.variables: first_file_data = first_file_data.rename({'valid_time':'time'})
        lat_chunk = 20
        N_lat_chunks = len(first_file_data.latitude) / lat_chunk
        lon_chunk = 100
        N_lon_chunks = len(first_file_data.longitude) / lon_chunk
        time_chunk = int(N_lat_chunks * N_lon_chunks * 0.0025 * len(first_file_data.time) / first_file_size)
        if time_chunk < 10 or time_chunk > 2500: 'Caution from ldp.load_era5(): revisit chunk length calculation'
        first_file_data.close()

    if process_and_export:
        # create reverse look-up table of variable abbreviations and long names (only necessary because files with both
        #   validated ERA5 and preliminary ERA5T data and a single variable [?] are missing the variable abbreviation)
        var_name_lookup = {}
        for filename in all_filenames:
            data = xr.open_dataset(data_dir + filename,chunks={})
            assert ('p0001' in data.variables) == ('p0005' in data.variables), \
                'Error from ldp.load_era5(): variables not recognized as either from validated ERA5 or preliminary ' \
                'ERA5T in filename {0}'.format(filename)
            if 'p0001' not in data.variables and 'p0005' not in data.variables:
                for var_abbrev in data.data_vars:
                    var_name_lookup[data[var_abbrev].long_name] = data[var_abbrev].name
            data.close()

        # process files containing both validated ERA5 and preliminary ERA5T data (merge the two and export)
        for f_idx, filename in enumerate(all_filenames):
            print('ldp.load_era5() is checking if file {0} of {1} needs to be merged'.format(f_idx+1,len(all_filenames)))
            data = xr.open_dataset(data_dir + filename,
                                   chunks={'valid_time':time_chunk,'time':time_chunk,'latitude':lat_chunk,'longitude':lon_chunk})
            if 'valid_time' in data.variables:
                data = data.rename({'valid_time':'time'})
            if 'expver' in data.variables:
                if data['expver'].dims[0] == 'expver':   # legacy format from GRIB-to-netCDF converter for files with both ERA5 and ERA5T data
                    data = data.sel(expver=1).combine_first(data.sel(expver=5))   # best way
                    # data = data.reduce(nansum,'expver')                         # more risky way
                    data.to_netcdf(data_dir + filename.rstrip('.nc') + '_ERA5_ERA5T_legacy_v2_merged.nc')
                    data.close()
                    if 'To delete' not in os.listdir(data_dir): os.mkdir(data_dir + 'To delete/')
                    _ = shutil.move(data_dir + filename,data_dir + 'To delete/' + filename)
                elif data['expver'].dims[0] == 'time':   # current format from GRIB-to-netCDF converter for files with both ERA5 and ERA5T data
                    data.close()
                else:
                    print('ERROR: ERA5 file is in an unrecognized format – check how merged ERA5/ERA5T data files are currently being handled by ECMWF')
                    data.close()
            elif 'p0001' in data.variables and 'p0005' in data.variables:   # likely only applies to super-legacy format for ERA5 files (two generations ago)
                era5_validated = data['p0001'].dropna('time',how='all')
                era5_validated.name = var_name_lookup[era5_validated.long_name]
                era5t_prelim = data['p0005'].dropna('time',how='all')
                era5t_prelim.name = var_name_lookup[era5t_prelim.long_name]
                dataarray = xr.concat([era5_validated,era5t_prelim],dim='time')
                dataarray \
                    = dataarray.chunk({'time':time_chunk,'latitude':lat_chunk,'longitude':lon_chunk})  # just in case
                # check for weird case where arrays at some times are filled with the add_offset value instead of NaNs
                if len(unique(dataarray.time)) != len(dataarray.time):
                    unique_times,unique_idx = unique(dataarray.time,return_index=True)
                    dataarray = dataarray.isel(time=unique_idx)  # ignore second occurrences of each non-unique time
                data = dataarray.to_dataset()
                data.to_netcdf(data_dir + filename.rstrip('.nc') + '_ERA5_ERA5T_legacy_v1_merged.nc')
                data.close()
                if 'To delete' not in os.listdir(data_dir): os.mkdir(data_dir + 'To delete/')
                _ = shutil.move(data_dir + filename,data_dir + 'To delete/' + filename)
            else:
                data.close()

        # split up files containing multiple variables (without this, open_mfdataset() will hang and cannot finish)
        all_filenames = os.listdir(data_dir)
        for filename in all_filenames:
            if '.nc' in filename:
                data = xr.open_dataset(data_dir + filename,
                                       chunks={'time':time_chunk,'latitude':lat_chunk,'longitude':lon_chunk})
                if len(data.data_vars) > 1:
                    for var_name in data.data_vars:
                        print('ldp.load_era5() is splitting variable {0} from file {1}'.format(var_name,filename))
                        data_subset = data[var_name].to_dataset()
                        data_subset.to_netcdf(data_dir + filename.rstrip('.nc') + '_split_{0}.nc'.format(var_name))
                    data.close()
                    if 'To delete' not in os.listdir(data_dir): os.mkdir(data_dir + 'To delete/')
                    _ = shutil.move(data_dir + filename,data_dir + 'To delete/' + filename)
        
        print('>>> ldp.era5() is finished!')
        return

    # load all files in directory
    # catch PerformanceWarning about "Increasing number of chunks by factor of X"; re-chunking makes this irrelevant
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',dask.array.PerformanceWarning)
        def era5_preprocess(ds):
            if 'valid_time' in ds.coords:
                ds = ds.rename({'valid_time':'time'})
            if 'number' in ds.coords:
                ds = ds.drop('number')
            if 'expver' in ds.variables:
                if ds['expver'].dims[0] == 'time' or ds['expver'].dims[0] == 'valid_time':
                    ds = ds.drop('expver')
            for grib_name in grib_rename.index:
                if grib_name in ds.variables:
                    ds = ds.rename({grib_name:grib_rename.loc[grib_name][grib_str + '_var']})
                    ds[grib_rename.loc[grib_name][grib_str + '_var']].attrs['long_name'] = grib_rename.loc[grib_name][grib_str + '_long']
            # ds = ds.chunk({'time':time_chunk,'latitude':lat_chunk,'longitude':lon_chunk})
            return ds
        all_data = xr.open_mfdataset(data_dir + '*.nc',combine='by_coords',preprocess=era5_preprocess,
                                     chunks={'valid_time':time_chunk,'time':time_chunk,'latitude':lat_chunk,'longitude':lon_chunk})
        
    # re-chunk (sometimes necessary when combining if some time indices have different lengths)
    if rechunk:
        all_data = all_data.chunk({'time':time_chunk,'latitude':lat_chunk,'longitude':lon_chunk})

    # rename for convenience
    if 'longitude' in all_data and 'latitude' in all_data:
        all_data = all_data.rename({'latitude':'lats','longitude':'lons'})

    # slice dimensions
    if datetime_range is not None:
        all_data = all_data.sel(time=slice(datetime_range[0],datetime_range[1]))
    if lat_range is not None:
        all_data = all_data.sel(lats=slice(lat_range[0],lat_range[1]))
    if lon_range is not None:
        all_data = all_data.sel(lons=slice(lon_range[0],lon_range[1]))

    for var_abbrev in all_data.data_vars:
        # rename some variable long names and units for convenience, e.g. during plotting
        all_data[var_abbrev].attrs['long_name'] = all_data[var_abbrev].long_name.replace('metre','m')
        all_data[var_abbrev].attrs['units'] \
            = all_data[var_abbrev].units.replace('**-3','^{-3}').replace('**-2','^{-2}').replace('**-1','^{-1}') \
                .replace('**2','^{2}').replace('**3','^{3}')

        # revise units for convenience and/or deaccumulate
        # note: this evaluates lazily using Dask, so expect processing hangs upon computation (instead of load)
        if var_abbrev == 'e' or var_abbrev == 'mer' or var_abbrev == 'avg_ie': all_data[var_abbrev] *= -1
        if all_data[var_abbrev].attrs['units'] == 'Pa':
            orig_name = all_data[var_abbrev].attrs['long_name']
            all_data[var_abbrev] /= 100.0
            all_data[var_abbrev].attrs = {'units':'hPa','long_name':orig_name}
        elif all_data[var_abbrev].attrs['units'] == 'K' and var_abbrev != 'd2m':
            orig_name = all_data[var_abbrev].attrs['long_name']
            all_data[var_abbrev] -= 273.15
            all_data[var_abbrev].attrs = {'units':'°C','long_name':orig_name}
        elif all_data[var_abbrev].attrs['units'] == 'J m^{-2}':   # deaccumulate
            orig_name = all_data[var_abbrev].attrs['long_name']
            all_data[var_abbrev] /= (60.0 * 60.0)
            all_data[var_abbrev].attrs = {'units':'W m^{-2}','long_name':orig_name}

    # add day-of-year as a secondary coordinate with dimension 'time'
    if 'doy' not in all_data.coords:
        datetime_index = pd.to_datetime(all_data['time'].values)
        doy_index = datetime_index.dayofyear + datetime_index.hour / 24. + datetime_index.minute / 60.
        all_data.coords['doy'] = ('time',doy_index)

    # calculate 10-m wind speed from u, v
    # note: this evaluates lazily using Dask, so expect processing hangs upon computation (instead of load)
    if 'si10' not in all_data and 'u10' in all_data and 'v10' in all_data:
        all_data['si10'] = (all_data['u10']**2 + all_data['v10']**2)**0.5
        all_data['si10'].attrs['units'] = 'm s^{-1}'
        all_data['si10'].attrs['long_name'] = '10 m wind speed'

    # calculate 2-m specific humidity from surface pressure and dewpoint temperature, if available
    # note: this evaluates lazily using Dask, so expect processing hangs upon computation (instead of load)
    # uses Equations 7.4 and 7.5 on p. 92 of ECMWF IFS Documentation, Ch. 7:
    #   https://www.ecmwf.int/sites/default/files/elibrary/2015/9211-part-iv-physical-processes.pdf
    if 'q2m' not in all_data and 'd2m' in all_data and 'msl' in all_data:
        # constants for Teten's formula for saturation water vapor pressure over water [not ice] (Eq. 7.5)
        # origin: Buck (1981)
        a1 = 611.21 # Pa
        a3 = 17.502 # unitless
        a4 = 32.19  # K
        T_0 = 273.16 # K

        # saturation water vapor pressure; units: Pa
        e_sat_at_Td = a1 * exp(a3 * (all_data['d2m'] - T_0) / (all_data['d2m'] - a4))

        # saturation specific humidity at dewpoint temperature (Eq. 7.4)
        # note conversion of surface pressure from hPa back to Pa
        R_dry_over_R_vap = 0.621981  # gas constant for dry air over gas constant for water vapor, p. 110
        q_sat_at_Td = R_dry_over_R_vap * e_sat_at_Td / (100*all_data['msl'] - (e_sat_at_Td*(1.0 - R_dry_over_R_vap)))

        all_data['q2m'] = q_sat_at_Td
        all_data['q2m'].attrs['units'] = 'kg kg^{-1}'
        all_data['q2m'].attrs['long_name'] = 'Specific humidity at 2 m'

    return all_data

## OC-CCI downloading routine

In [ ]:
# download all monthly OC-CCI ocean color data (Sep. 1997 to present)
# (NOTE: took about ~5 hours on Deep server)
for d in pd.date_range('1997-09-01','2026-06-01',freq='MS'):
    mo_str = f'{d.year}{d.month:02d}'
    filename = f'ESACCI-OC-L3S-CHLOR_A-MERGED-1M_MONTHLY_4km_GEO_PML_OCx-{mo_str}-fv6.0.nc'
    url_start = f'https://www.oceancolour.org/thredds/fileServer/cci/v6.0-release/geographic/monthly/chlor_a/{d.year}/'
    single_file(url_start,filename,chl_orig_dir,ftp_root=False,overwrite=False)
    print(f'>>> OC-CCI data for {mo_str} downloaded')

## ERA5 downloading/processing routine

To copy new files or file versions from Linux server to local machine, use rsync:

rsync -avzh user@server.address:"<remote_path>" "<local_path>"

rsync -avzh ethancc@deep.ocean.washington.edu:"/dat1/ethancc/Data/Sea\ ice\ drift/NSIDC_Pathfinder/" "/Users/Ethan/Documents/Research/2016-08 - UW/Data/Sea ice drift/NSIDC_Pathfinder"
In remote path, escape spaces with backslashes and include trailing forward slash at end of directory name
In local path, no need to escape spaces with backslashes or include trailing forward slash

In [5]:
download_era5 = False
process_era5 = False

# download ERA5 reanalysis data
# (this initiates downloads for ECMWF ERA5 reanalysis fields and processes them after downloading)
# - IMPORTANT: after ECMWF's migration to a new GRIB to netCDF converter in late 2024, xarray I/O with the resulting netCDF4 files is extremely slow
#   compared to the previous netCDF3 files; see ldp.load_era5() for more details; hopefully this gets fixed, but in the interim, this routine
#   submits requests for the legacy netCDF3 files
# - NOTE: if larger requests that include many years fail, check Request Size validation, noting that netCDF size limits have decreased
#   (see: https://forum.ecmwf.int/t/limitation-change-on-netcdf-era5-requests/12477)
# - NOTE: this routine still uses GRIB1 naming conventions; variables that were renamed in ECMWF's migration to GRIB2 may eventually fail to download
#   but seem to be working fine for now, though as the migration proceeds more variables may need to be added to the rename list in ldp.era5()
#   (see status here: https://confluence.ecmwf.int/display/MTG2US/Changes+in+ecCodes+version+2.39.0+compared+to+the+previous+version)
# - download processed files from: https://cds.climate.copernicus.eu/requests?tab=all
# - if downloading to Linux, download in background using wget -bqc <URL>
# - if downloading to MacOS, consider using a stand-alone download manager app
# - names of key variables not downloaded are listed below for future reference:
if download_era5:
    variables = ['10m_u_component_of_wind',
                 '10m_v_component_of_wind',
                 'sea_surface_temperature']
    for var in variables:
        print('>>> submitting CDS API requests to retrieve <{0}>'.format(var))
        era5(area=[-50,-180,-80,180],
             years=[str(yr) for yr in range(1997,2004+1)],
             variables=[var],
             batch=True,legacy_netcdf=True)
        era5(area=[-50,-180,-80,180],
             years=[str(yr) for yr in range(2005,2014+1)],
             variables=[var],
             batch=True,legacy_netcdf=True)
        era5(area=[-50,-180,-80,180],
             years=[str(yr) for yr in range(2015,2024+1)],
             variables=[var],
             batch=True,legacy_netcdf=True)
        era5(area=[-50,-180,-80,180],
             years=['2025'],
             variables=[var],
             batch=True,legacy_netcdf=True)   # 2025 will be a mix of ERA5 and preliminary ERA5T fields (double the size per year of data)
        
# process ERA5 reanalysis data
# - for data files with both ERA5 and preliminary ERA5T data, this creates new data files that merge the two
# - for data files with more than one variable (if applicable), this exports variables to separate files
# - note: delete files in 'To delete' folder after running
if process_era5:
    load_era5(era5_dir,process_and_export=True)

ldp.load_era5() is checking if file 1 of 12 needs to be merged
ldp.load_era5() is checking if file 2 of 12 needs to be merged
ldp.load_era5() is checking if file 3 of 12 needs to be merged
ldp.load_era5() is checking if file 4 of 12 needs to be merged
ldp.load_era5() is checking if file 5 of 12 needs to be merged
ldp.load_era5() is checking if file 6 of 12 needs to be merged
ldp.load_era5() is checking if file 7 of 12 needs to be merged
ldp.load_era5() is checking if file 8 of 12 needs to be merged
ldp.load_era5() is checking if file 9 of 12 needs to be merged
ldp.load_era5() is checking if file 10 of 12 needs to be merged
ldp.load_era5() is checking if file 11 of 12 needs to be merged
ldp.load_era5() is checking if file 12 of 12 needs to be merged
>>> ldp.era5() is finished!


In [4]:
# load ERA5 reanalysis (concatenate and process parameters lazily using Dask)
era5 = load_era5(era5_dir,use_grib2_names=False,rechunk=True)

# trim ERA5 to account for slightly different end times of different data files (which depends on when each file was served by ECMWF)
# - trim the final 72 hours to be on the safe side
era5 = era5.isel(time=slice(None,-72))

In [6]:
# create new variable representing U^3 (wind speed to the third power, approximately proportional to power input)
era5['si10_cubed'] = era5['si10']**3

# (lazily) calculate monthly means, with computation to be triggered upon file export
era5_monthly_means = era5.resample(time='1MS').mean()
year_list = unique(era5_monthly_means['time'].dt.year)

In [ ]:
# export monthly means to new netCDF file (ignore variables for u- and v-components of wind)
for year in year_list:
    print(f"Working on year {year}")
    with ProgressBar():
        era5_monthly_means[['si10','si10_cubed','sst']].sel(time=str(year)).to_netcdf(era5_mo_means_dir + f'ERA5_SURP_monthly_means_{year}.nc')